# Lab 4: GAN Training Tricks — Break It, Then Fix It

**Goal:** Deliberately cause GAN training failures, then apply tricks to fix them.

**What you'll learn:**
- How to recognize mode collapse (and cause it on purpose)
- How to detect vanishing gradients (D wins too easily)
- Label smoothing — what it does and why it helps
- Instance noise — handicapping the Discriminator
- Learning rate balance — the most important trick
- The GAN training checklist for real projects

**Philosophy:** The best way to learn what breaks is to break it yourself.

**Time:** ~75 minutes. This lab trains ~15 small DCGANs, so the setup cell picks a training-set size per device (`TRAIN_SUBSET`). Approximate total compute:

| Device | Images per epoch | Total |
|---|---|---|
| NVIDIA GPU (Colab T4) | all 60k | ~30 min |
| Apple-silicon laptop (`mps`) | 20k | ~20–25 min |
| CPU only | 5k | ~45–50 min |

Short on time (CPU)? Run Parts 2–4 and the Part 10 challenge; skip Parts 6–7.

**How to use this lab:** every experiment starts with a **Predict** prompt — write your answer down before running.

---

## Part 1: Setup — Our Baseline DCGAN

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else 'cpu')
print(f"Using device: {device}")

# Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Many runs ahead: use a subset per device so the lab fits the session (None = all 60k images).
TRAIN_SUBSET = {'cuda': None, 'mps': 20_000}.get(device.type, 5_000)
if TRAIN_SUBSET:
    dataset = torch.utils.data.Subset(dataset, range(TRAIN_SUBSET))
print(f"Training images per epoch: {len(dataset):,}")
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
def make_generator(noise_size=100):
    return nn.Sequential(
        nn.Linear(noise_size, 256 * 7 * 7), nn.ReLU(),
        nn.Unflatten(1, (256, 7, 7)),
        nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.ConvTranspose2d(128, 1, 4, 2, 1), nn.Tanh()
    ).to(device)

def make_discriminator():
    return nn.Sequential(
        nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
        nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
        nn.Flatten(),
        nn.Linear(128 * 7 * 7, 1), nn.Sigmoid()
    ).to(device)

In [ ]:
def train_gan(gen, dis, dataloader, epochs=10,
              lr_g=0.0002, lr_d=0.0002,
              noise_size=100,
              label_smooth_real=1.0, label_smooth_fake=0.0,
              instance_noise=0.0,
              d_steps=1, g_steps=1,
              name="GAN"):
    """
    Flexible GAN trainer with knobs for all the tricks.
    Returns training history for analysis.
    """
    loss_fn = nn.BCELoss()
    g_opt = torch.optim.Adam(gen.parameters(), lr=lr_g, betas=(0.5, 0.999))
    d_opt = torch.optim.Adam(dis.parameters(), lr=lr_d, betas=(0.5, 0.999))

    history = {
        'g_loss': [], 'd_loss': [],
        'd_real': [], 'd_fake': [],
        'images': {}  # snapshots at certain epochs
    }

    fixed_noise = torch.randn(16, noise_size, device=device)

    for epoch in range(1, epochs + 1):
        eg, ed, edr, edf, nb = 0, 0, 0, 0, 0

        for real_images, _ in dataloader:
            bs = real_images.size(0)
            real = real_images.to(device)

            # Add instance noise to real images (trick)
            if instance_noise > 0:
                real = real + instance_noise * torch.randn_like(real)

            # Labels with optional smoothing
            real_labels = torch.ones(bs, 1, device=device) * label_smooth_real
            fake_labels = torch.zeros(bs, 1, device=device) + label_smooth_fake

            # ====== Train Discriminator (d_steps times) ======
            for _ in range(d_steps):
                d_loss_real = loss_fn(dis(real), real_labels)

                noise = torch.randn(bs, noise_size, device=device)
                fake = gen(noise).detach()
                if instance_noise > 0:
                    fake = fake + instance_noise * torch.randn_like(fake)
                d_loss_fake = loss_fn(dis(fake), fake_labels)

                d_loss = d_loss_real + d_loss_fake
                d_opt.zero_grad(); d_loss.backward(); d_opt.step()

            # ====== Train Generator (g_steps times) ======
            for _ in range(g_steps):
                noise = torch.randn(bs, noise_size, device=device)
                fake = gen(noise)
                g_loss = loss_fn(dis(fake), torch.ones(bs, 1, device=device))
                g_opt.zero_grad(); g_loss.backward(); g_opt.step()

            # Track
            with torch.no_grad():
                d_on_real = dis(real_images.to(device)).mean().item()
                d_on_fake = dis(gen(torch.randn(bs, noise_size, device=device))).mean().item()

            eg += g_loss.item()
            ed += d_loss.item()
            edr += d_on_real
            edf += d_on_fake
            nb += 1

        history['g_loss'].append(eg / nb)
        history['d_loss'].append(ed / nb)
        history['d_real'].append(edr / nb)
        history['d_fake'].append(edf / nb)

        # Save image snapshots
        if epoch in [1, epochs // 2, epochs]:
            gen.eval()
            with torch.no_grad():
                history['images'][epoch] = gen(fixed_noise).cpu()
            gen.train()

        if epoch % max(1, epochs // 5) == 0 or epoch == 1:
            print(f"  [{name}] Epoch {epoch:2d}: "
                  f"D={history['d_loss'][-1]:.4f} G={history['g_loss'][-1]:.4f} "
                  f"D(real)={history['d_real'][-1]:.3f} D(fake)={history['d_fake'][-1]:.3f}")

    return history

In [ ]:
def plot_comparison(results, title):
    """Plot training curves and sample images for multiple experiments."""
    n = len(results)
    fig, axes = plt.subplots(n, 4, figsize=(20, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for row, (name, hist) in enumerate(results.items()):
        # Col 0: Losses
        axes[row, 0].plot(hist['g_loss'], label='G', color='blue', linewidth=2)
        axes[row, 0].plot(hist['d_loss'], label='D', color='red', linewidth=2)
        axes[row, 0].set_title(f'{name}\nLosses', fontsize=11)
        axes[row, 0].legend(fontsize=9)
        axes[row, 0].grid(True, alpha=0.3)
        axes[row, 0].set_xlabel('Epoch')

        # Col 1: D scores
        axes[row, 1].plot(hist['d_real'], label='D(real)', color='green', linewidth=2)
        axes[row, 1].plot(hist['d_fake'], label='D(fake)', color='orange', linewidth=2)
        axes[row, 1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
        axes[row, 1].set_title('D Confidence', fontsize=11)
        axes[row, 1].legend(fontsize=9)
        axes[row, 1].set_ylim(0, 1)
        axes[row, 1].grid(True, alpha=0.3)
        axes[row, 1].set_xlabel('Epoch')

        # Col 2-3: Generated images (first and last epoch)
        epochs_saved = sorted(hist['images'].keys())
        for col_offset, ep in enumerate([epochs_saved[0], epochs_saved[-1]]):
            imgs = hist['images'][ep]
            # Create 2x4 grid in the subplot
            grid = []
            for r in range(2):
                row_imgs = []
                for c in range(4):
                    idx = r * 4 + c
                    row_imgs.append(imgs[idx, 0].numpy())
                grid.append(np.concatenate(row_imgs, axis=1))
            grid = np.concatenate(grid, axis=0)
            axes[row, 2 + col_offset].imshow(grid, cmap='gray', vmin=-1, vmax=1)
            axes[row, 2 + col_offset].set_title(f'Epoch {ep}', fontsize=11)
            axes[row, 2 + col_offset].axis('off')

    plt.suptitle(title, fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## Part 2: Failure Mode 1 — Mode Collapse

**What is it?** The Generator finds ONE output that fools D and keeps producing only that.

**How to cause it:**
- Very small noise vector (z=2 instead of 100)
- Train G much more than D

**How to spot it:**
- All generated images look the same
- G loss is low, but there's no variety

**Predict:** which run will have the *lowest* diversity score — z=2, or 5 G-steps per D-step?

In [ ]:
print("=" * 60)
print("EXPERIMENT: Causing Mode Collapse")
print("=" * 60)

# Healthy baseline
print("\n--- Healthy GAN (z=100, balanced training) ---")
gen_healthy = make_generator(noise_size=100)
dis_healthy = make_discriminator()
hist_healthy = train_gan(gen_healthy, dis_healthy, dataloader,
                         epochs=10, noise_size=100, name="Healthy")

# Mode collapse: tiny noise vector
print("\n--- Collapsed GAN (z=2, forces limited variety) ---")
gen_collapsed = make_generator(noise_size=2)
dis_collapsed = make_discriminator()
hist_collapsed = train_gan(gen_collapsed, dis_collapsed, dataloader,
                           epochs=10, noise_size=2, name="Collapsed")

# Mode collapse: train G way more than D
print("\n--- Over-trained G (5 G steps per 1 D step) ---")
gen_overtrain = make_generator(noise_size=100)
dis_overtrain = make_discriminator()
hist_overtrain = train_gan(gen_overtrain, dis_overtrain, dataloader,
                           epochs=10, noise_size=100,
                           d_steps=1, g_steps=5, name="G-Overtrained")

In [ ]:
plot_comparison({
    'Healthy (z=100)': hist_healthy,
    'Mode Collapse (z=2)': hist_collapsed,
    'G Overtrained (5G:1D)': hist_overtrain,
}, title='Failure Mode 1: Mode Collapse')

In [ ]:
# Diversity check: generate many images and measure variety
def diversity_score(generator, noise_size, n=100):
    """Measure how diverse the generated images are."""
    generator.eval()
    with torch.no_grad():
        imgs = generator(torch.randn(n, noise_size, device=device)).cpu().view(n, -1)
    generator.train()

    # Average pairwise distance between generated images
    dists = torch.cdist(imgs, imgs)
    mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
    avg_dist = dists[mask].mean().item()
    return avg_dist

print("Diversity scores (higher = more variety):")
print(f"  Healthy (z=100):      {diversity_score(gen_healthy, 100):.4f}")
print(f"  Collapsed (z=2):      {diversity_score(gen_collapsed, 2):.4f}")
print(f"  G-Overtrained:        {diversity_score(gen_overtrain, 100):.4f}")

print("\nLow diversity = mode collapse. All outputs look the same.")

### Takeaway: Mode Collapse

*Note:* z=2 does not always collapse on MNIST (2 numbers can still encode many digits) — check the diversity scores, not just the pictures.

| Cause | Fix |
|---|---|
| Noise vector too small | Use z=100 or z=128 |
| G trains much more than D | Keep 1:1 ratio (or train D slightly more) |
| Poor loss function | Use WGAN loss (Unit 2, L08) |

---

## Part 3: Failure Mode 2 — D Wins Too Easily (Vanishing Gradients)

**What is it?** D becomes too strong too fast. It outputs 0 for ALL fakes with 100% confidence. G gets zero gradient signal.

**How to cause it:**
- Give D a much higher learning rate
- Train D many more steps than G

**How to spot it:**
- D_loss → 0
- G_loss stays high and flat
- D(fake) = 0.000
- Generated images stop improving

**Predict:** with D's learning rate 50× higher, what will D(fake) be after 10 epochs? And G_loss?

In [ ]:
print("=" * 60)
print("EXPERIMENT: D Wins Too Easily")
print("=" * 60)

# Healthy baseline (reuse from above)
print("\n--- Healthy (balanced LR) ---")
# Already have hist_healthy

# D too strong: much higher LR
print("\n--- D too strong (D lr=0.01, G lr=0.0002) ---")
gen_d_strong = make_generator()
dis_d_strong = make_discriminator()
hist_d_strong = train_gan(gen_d_strong, dis_d_strong, dataloader,
                          epochs=10, lr_g=0.0002, lr_d=0.01,
                          name="D-TooStrong")

# D too strong: 5 D steps per G step
print("\n--- D overtrained (5D:1G) ---")
gen_d_over = make_generator()
dis_d_over = make_discriminator()
hist_d_over = train_gan(gen_d_over, dis_d_over, dataloader,
                        epochs=10, d_steps=5, g_steps=1,
                        name="D-Overtrained")

In [ ]:
plot_comparison({
    'Healthy': hist_healthy,
    'D too strong (lr=0.01)': hist_d_strong,
    'D overtrained (5D:1G)': hist_d_over,
}, title='Failure Mode 2: Discriminator Wins Too Easily')

### Takeaway: D Wins Too Easily

| Symptom | What's happening |
|---|---|
| D_loss → 0 | D perfectly classifies everything |
| G_loss stays high | G gets no useful gradient signal |
| D(fake) = 0.000 | D is 100% confident all fakes are fake |

| Fix | How |
|---|---|
| Lower D learning rate | Slow D down |
| Label smoothing | Make D less confident |
| Instance noise | Add noise to inputs, make D's job harder |

---

## Part 4: Fix 1 — Label Smoothing

Instead of telling D that real = 1.0 and fake = 0.0, we use:
- Real labels: **0.9** instead of 1.0
- Fake labels: **0.1** instead of 0.0

This prevents D from becoming **too confident**, which keeps gradients flowing to G.

> The slides (L06) use this two-sided version. Salimans et al. (2016) recommend **one-sided** smoothing — only real = 0.9, fake stays 0.0 — because a positive target for fakes rewards G for staying where it is. Try it: `label_smooth_real=0.9, label_smooth_fake=0.0`.

**Predict:** will smoothing alone rescue the run with D's lr = 0.01?

In [ ]:
print("=" * 60)
print("FIX 1: Label Smoothing")
print("=" * 60)

# Broken: D too strong -- identical settings to 'D too strong' in Part 3, so reuse that run
# (saves one full training run)
print("\n--- Broken: D too strong (lr_d=0.01) -- reusing the Part 3 run ---")
hist_broken = hist_d_strong

# Fixed with label smoothing
print("\n--- Fixed: Label smoothing (real=0.9, fake=0.1) ---")
gen_smooth = make_generator()
dis_smooth = make_discriminator()
hist_smooth = train_gan(gen_smooth, dis_smooth, dataloader,
                        epochs=10, lr_g=0.0002, lr_d=0.01,
                        label_smooth_real=0.9, label_smooth_fake=0.1,
                        name="Label-Smoothed")

In [ ]:
plot_comparison({
    'Broken (D too strong)': hist_broken,
    'Fixed (Label Smoothing)': hist_smooth,
}, title='Fix 1: Label Smoothing Helps When D Is Too Strong')

### How label smoothing works:

```
Without smoothing:  D targets = [1.0, 0.0]  --> D can be 100% confident
With smoothing:     D targets = [0.9, 0.1]  --> D can never be more than 90% confident

Result: D's confidence is capped, gradients keep flowing to G
```

---

## Part 5: Fix 2 — Instance Noise

Add random noise to the images that D sees. This makes D's job harder, preventing it from becoming too strong.

It's like making the Discriminator wear slightly foggy glasses.

**Predict:** σ = 0.1 is applied to both real and fake images. Will that be enough against a D with lr = 0.01?

In [ ]:
# Visualize what instance noise does
sample_img = dataset[0][0]
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5]

fig, axes = plt.subplots(1, len(noise_levels), figsize=(15, 3))
for ax, sigma in zip(axes, noise_levels):
    noisy = sample_img + sigma * torch.randn_like(sample_img)
    ax.imshow(noisy[0], cmap='gray')
    ax.set_title(f'noise={sigma}', fontsize=11)
    ax.axis('off')

plt.suptitle('Instance Noise: adding Gaussian noise to images D sees',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("FIX 2: Instance Noise")
print("=" * 60)

# Broken: D too strong
print("\n--- Broken: D too strong ---")
# Reuse hist_broken from above

# Fixed with instance noise
print("\n--- Fixed: Instance noise (sigma=0.1) ---")
gen_noisy = make_generator()
dis_noisy = make_discriminator()
hist_noisy = train_gan(gen_noisy, dis_noisy, dataloader,
                       epochs=10, lr_g=0.0002, lr_d=0.01,
                       instance_noise=0.1,
                       name="Instance-Noise")

In [ ]:
plot_comparison({
    'Broken (D too strong)': hist_broken,
    'Fixed (Instance Noise)': hist_noisy,
}, title='Fix 2: Instance Noise Handicaps the Discriminator')

---
## Part 6: Fix 3 — Learning Rate Balance

The **single most important trick**: keep G and D learning rates balanced.

The DCGAN paper recommends `lr=0.0002` for both. Let's test different ratios.

**Predict:** rank the four settings from best to worst images.

In [ ]:
print("=" * 60)
print("FIX 3: Learning Rate Balance")
print("=" * 60)

lr_experiments = {
    'Balanced (0.0002/0.0002)': (0.0002, 0.0002),
    'D too fast (0.0002/0.002)': (0.0002, 0.002),
    'G too fast (0.002/0.0002)': (0.002, 0.0002),
    'Slow D fix (0.0002/0.0001)': (0.0002, 0.0001),
}

lr_results = {}
for name, (lr_g, lr_d) in lr_experiments.items():
    print(f"\n--- {name} ---")
    g = make_generator()
    d = make_discriminator()
    lr_results[name] = train_gan(g, d, dataloader,
                                 epochs=10, lr_g=lr_g, lr_d=lr_d,
                                 name=name)

In [ ]:
plot_comparison(lr_results,
                title='Fix 3: Learning Rate Balance is Critical')

### Takeaway: Learning Rate

| Scenario | Result |
|---|---|
| Balanced LR | Stable training, good images |
| D much faster | D dominates, G stuck (vanishing gradients) |
| G much faster | G "cheats", possible mode collapse |
| Slow D | D can't dominate, G gets better signal |

**Rule of thumb:** If training is unstable, try **lowering D's learning rate** first.

---

## Part 7: Combining Multiple Tricks

In practice, you combine tricks. Let's see the cumulative effect.

**Predict:** with already-healthy settings (lr 2e-4 both), will the tricks visibly improve the images, or mainly change D's confidence curves?

In [ ]:
print("=" * 60)
print("COMBINED TRICKS")
print("=" * 60)

# No tricks
print("\n--- No tricks ---")
gen_none = make_generator()
dis_none = make_discriminator()
hist_none = train_gan(gen_none, dis_none, dataloader,
                      epochs=15, name="No tricks")

# All tricks combined
print("\n--- All tricks: label smooth + instance noise + balanced LR ---")
gen_all = make_generator()
dis_all = make_discriminator()
hist_all = train_gan(gen_all, dis_all, dataloader,
                     epochs=15,
                     label_smooth_real=0.9,
                     label_smooth_fake=0.1,
                     instance_noise=0.05,
                     name="All tricks")

In [ ]:
plot_comparison({
    'No Tricks': hist_none,
    'All Tricks Combined': hist_all,
}, title='Combined Effect of All Tricks')

---
## Part 8: The Diagnostic Dashboard

In real projects, you monitor these during training. Here's how to build a diagnostic view.

In [ ]:
def diagnose_training(history, name):
    """Print a diagnosis of GAN training health."""
    g = history['g_loss']
    d = history['d_loss']
    dr = history['d_real']
    df = history['d_fake']

    print(f"\n{'=' * 50}")
    print(f"DIAGNOSIS: {name}")
    print(f"{'=' * 50}")

    # Check for D domination
    if d[-1] < 0.1:
        print("[WARNING] D_loss near 0 -- Discriminator is dominating!")
        print("  Fix: Lower D learning rate, add label smoothing")
    elif d[-1] > 3.0:
        print("[WARNING] D_loss very high -- Discriminator is struggling")

    # Check for G stuck
    if g[-1] > 5.0:
        print("[WARNING] G_loss very high -- Generator is stuck!")
        print("  Fix: Lower D lr, add instance noise")

    # Check D confidence
    if dr[-1] > 0.95:
        print("[WARNING] D(real) > 0.95 -- D too confident on real")
        print("  Fix: Label smoothing (real=0.9)")
    if df[-1] < 0.05:
        print("[WARNING] D(fake) < 0.05 -- D too confident on fake")
        print("  Fix: Instance noise, lower D lr")

    # Check for mode collapse (G loss low but flat)
    g_std = np.std(g[-5:])
    if g[-1] < 1.0 and g_std < 0.05:
        print("[WARNING] G_loss low and stable -- possible mode collapse")
        print("  Fix: Increase noise size, check image variety")

    # Check for oscillation
    d_changes = [abs(d[i] - d[i-1]) for i in range(1, len(d))]
    if np.mean(d_changes) > 0.5:
        print("[WARNING] Large loss swings -- training is oscillating")
        print("  Fix: Lower learning rate")

    # Healthy check
    if 0.3 < dr[-1] < 0.9 and 0.1 < df[-1] < 0.7:
        print("[OK] D confidence looks balanced")
    if abs(g[-1] - d[-1]) < 1.0:
        print("[OK] G and D losses are roughly balanced")

    print(f"\n  Final stats: G_loss={g[-1]:.4f} D_loss={d[-1]:.4f}")
    print(f"  D(real)={dr[-1]:.3f}  D(fake)={df[-1]:.3f}")

# Run diagnosis on our experiments
diagnose_training(hist_healthy, "Healthy Baseline")
diagnose_training(hist_d_strong, "D Too Strong")
diagnose_training(hist_collapsed, "Mode Collapse (z=2)")
diagnose_training(hist_all, "All Tricks Combined")

---
## Part 9: The GAN Training Checklist

Use this for every GAN you train:

```
+------------------------------------------------------------+
|              GAN TRAINING CHECKLIST                          |
|                                                              |
|  DATA                                                        |
|  [ ] Normalize images to [-1, 1]                             |
|  [ ] Shuffle dataset                                         |
|                                                              |
|  ARCHITECTURE                                                |
|  [ ] Generator output: Tanh                                  |
|  [ ] Generator hidden: ReLU                                  |
|  [ ] Discriminator hidden: LeakyReLU(0.2)                    |
|  [ ] Discriminator output: Sigmoid                           |
|  [ ] BatchNorm in both (NOT in D's first layer)              |
|  [ ] Mirror architecture: G upsamples, D downsamples         |
|                                                              |
|  TRAINING                                                    |
|  [ ] Adam optimizer: lr=0.0002, betas=(0.5, 0.999)           |
|  [ ] Label smoothing: real=0.9, fake=0.1                     |
|  [ ] Same number of D and G steps (1:1)                      |
|  [ ] .detach() fake images when training D                   |
|                                                              |
|  MONITORING                                                  |
|  [ ] Track D_loss and G_loss (neither should go to 0)        |
|  [ ] Track D(real) and D(fake) scores                        |
|  [ ] Save generated images every N epochs                    |
|  [ ] Check variety (not all outputs the same)                |
|                                                              |
|  IF STUCK                                                    |
|  [ ] D too strong? Lower D lr, add label smoothing           |
|  [ ] Mode collapse? Increase noise size, check D:G ratio     |
|  [ ] Oscillating? Lower both learning rates                  |
|  [ ] Blurry? Switch to convolutional (DCGAN)                 |
+------------------------------------------------------------+
```

---
## Part 10: Your Turn — Fix a Broken GAN

Here's a deliberately broken GAN. Your job: diagnose the problem and fix it.

In [ ]:
# CHALLENGE: This GAN is broken. Run it, diagnose the problem, and fix it.

gen_broken_challenge = make_generator(noise_size=100)
dis_broken_challenge = make_discriminator()

# Something is wrong with these settings...
hist_challenge = train_gan(
    gen_broken_challenge,
    dis_broken_challenge,
    dataloader,
    epochs=10,
    lr_g=0.00005,     # <-- G learning rate
    lr_d=0.005,       # <-- D learning rate (100x faster!)
    d_steps=3,        # <-- D trains 3x more
    g_steps=1,
    name="BROKEN"
)

diagnose_training(hist_challenge, "Broken Challenge")

In [ ]:
# YOUR FIX: Modify the settings below to fix the GAN
# Hint: Look at the diagnosis above.
#       What's wrong? D learning rate? Number of steps? Missing tricks?
# The values below are ONE reference fix -- cover them up and try your own first.

gen_fixed_challenge = make_generator(noise_size=100)
dis_fixed_challenge = make_discriminator()

hist_fixed = train_gan(
    gen_fixed_challenge,
    dis_fixed_challenge,
    dataloader,
    epochs=10,
    lr_g=0.0002,          # TODO: fix this
    lr_d=0.0002,          # TODO: fix this
    d_steps=1,            # TODO: fix this
    g_steps=1,
    label_smooth_real=0.9,  # TODO: try adding label smoothing
    label_smooth_fake=0.1,
    name="YOUR FIX"
)

diagnose_training(hist_fixed, "Your Fix")

In [ ]:
# Compare broken vs your fix
plot_comparison({
    'BROKEN': hist_challenge,
    'YOUR FIX': hist_fixed,
}, title='Challenge: Can You Fix the Broken GAN?')

---
## Part 11: Summary

| Failure | Symptoms | Fix |
|---|---|---|
| **Mode collapse** | All outputs identical, low diversity | Increase z, balance D:G steps |
| **D too strong** | D_loss→0, G_loss stuck high, D(fake)=0 | Label smoothing, instance noise, lower D lr |
| **Oscillation** | Loss swings wildly | Lower both learning rates |
| **Vanishing gradients** | G can't learn, images stuck | Non-saturating loss (already used here: G is trained with real labels), instance noise |

| Trick | What it does |
|---|---|
| **Label smoothing** | Caps D's confidence (0.9/0.1 instead of 1.0/0.0) |
| **Instance noise** | Adds noise to D's input, makes classification harder |
| **LR balance** | Keep G and D learning at similar speeds |
| **BatchNorm** | Keeps internal values stable |
| **Monitor both losses** | Neither should go to 0 |

**The most important lesson:** GAN training is a **balancing act**. Neither network should dominate. Monitor, diagnose, and adjust.

**Next (Unit 2):** L07 explains why these tricks are band-aids; L08's WGAN replaces BCE loss with Wasserstein distance — a fundamentally better way to train GANs that avoids many of these problems.